# 07 | Revisão técnica de versões explícitas

POC semissintética • contrato provisório • sem homologação acadêmica.
Execute em kernel reiniciado, de cima para baixo. As credenciais vêm do ambiente.


**Objetivo:** inspecionar candidatas e aprovar conscientemente. **Entradas:** campanha04/05; credencial de operador por ambiente. Por padrão apenas inspeciona. **Esperado:** estados distintos VALIDATED/APPROVED, sem homologação acadêmica.

In [ ]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.application_client import call, context, save_context

from src.workbench.academic_ui import campaign_result,model_table
result=campaign_result()
display(model_table(result['models']))
APPROVE_DEMO=os.getenv('V4_APPROVE_DEMO','false').lower()=='true'
REASON='Revisão técnica explícita para demonstração POC sintética; sem homologação acadêmica.'


**Executar:** solicitação e revisão autenticadas. Para aprovar, defina `APPROVE_DEMO=True` e reexecute esta célula. **Resultado:** SQL e alias reconciliados; erros deixam o modelo bloqueado.

In [ ]:

if APPROVE_DEMO:
    reviews=call('GET','/v1/governance/requests?limit=500')
    for model in result['models']:
        selector={k:model[k] for k in ('model_name','model_version')}
        current=[r for r in reviews if all(str(r[k])==str(v) for k,v in selector.items())]
        if any(r['status']=='APPROVED' for r in current):
            continue
        pending=[r for r in current if r['status'] in ('RECONCILING','REQUESTED') and r['action']=='APPROVE']
        request=pending[0] if pending else call('POST','/v1/governance/requests',dict(**selector,action='APPROVE',reason=REASON,requested_by='jupyter-poc-reviewer'))
        call('POST',f"/v1/governance/requests/{request['approval_id']}/review",dict(approve=True,confirm='REVIEW_EXPLICIT_VERSION'),operator=True)
else:
    display('INSPECT_ONLY: nenhuma aprovação executada.')
display(pd.DataFrame(call('GET','/v1/serving/models')))


**Onde conferir:** [MLflow](http://localhost:5000), [RustFS](http://localhost:9001), [API](http://localhost:8000/docs), [Logs](http://localhost:8088). Em falha: `docker compose logs --tail 100 api inference worker mlflow`.